In [121]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import Chroma
import os

from langchain_community.llms import Ollama
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

import logging
import json



import shutil
from datetime import datetime
import chromadb
from chromadb.config import Settings

In [122]:
DATA_PATH = "../data/documents"
VECTOR_DB_PATH = "../data/embeddings"
EMBED_MODEL = "nomic-embed-text"
METADATA_FILE = "../data/document_metadata.json"
LOG_FILE = "../data/document_ingestion.log"


logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(LOG_FILE),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

In [123]:
class RAG_Manager:
    def __init__(self, docs_path=DATA_PATH, embed_path=VECTOR_DB_PATH, embed_model_name=EMBED_MODEL, chunk_size=1000, chunk_overlap=100):
        self.docs_path = docs_path
        self.embed_path = embed_path
        self.embed_model_name = embed_model_name
        self.embed_model = OllamaEmbeddings(model=self.embed_model_name)
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            add_start_index=True,
        )
        self.collection = None
        self.init_chroma()
    
    def init_chroma(self):
        """Initialize Chroma vector store."""
        shutil.rmtree(self.embed_path, ignore_errors=True)
        if not os.path.exists(self.embed_path):
            os.makedirs(self.embed_path)
        
        chroma_client = chromadb.PersistentClient(
            path=self.embed_path,
            settings=Settings(persist_directory=self.embed_path),
            tenant="default_tenant",
            database="default_database"
        )
        self.collection = chroma_client.get_or_create_collection(name="my_collection")
        return chroma_client

    
    def chunkify(self, file):
        """Split text into chunks with overlap."""
        if file.endswith(".pdf"):
            content = PyPDFLoader(os.path.join(self.docs_path, file)).load()
            full_text = " ".join(doc.page_content for doc in content)
            source_file = content[0].metadata['source']
            return self.text_splitter.split_text(full_text), file
        return [], 'None'
    def get_files(self):
        """Get all files in the documents directory."""
        files = []
        for file in sorted(os.listdir(self.docs_path)):
            if file.endswith(".pdf"):
                files.append(file)
        return files
    
    def get_metadata(self):
        """Load metadata from the JSON file."""
        if not os.path.exists(METADATA_FILE):
            return {}
        with open(METADATA_FILE, 'r') as f:
            metadata = json.load(f)
        return metadata
    
    def save_metadata(self, metadata):
        """Save metadata to the JSON file."""
        with open(METADATA_FILE, 'w') as f:
            json.dump(metadata, f, indent=4)
    
    def process_file(self, file):
        """Read, chunkify and embed file content"""
        text, source = self.chunkify(file)
        # print(text[500:600])
        embeddings = self.embed_model.embed_documents(text[:100])
        # embeddings = []
        return embeddings, source

    def ingest_documents(self):
        available_documents = self.get_files()
        metadata_store = self.get_metadata()

        current_documents = self.get_metadata().keys()
        new_documents = set(available_documents) - set(current_documents)
        docs_to_remove = set(current_documents) - set(available_documents)

        if not new_documents:
            logger.info("No new documents to ingest.")
        else:
            logger.info(f"Ingesting {len(new_documents)} new documents.")
        if docs_to_remove:
            logger.info(f"Deleting {len(docs_to_remove)} unavailable documents.")

        while new_documents:
            embeddings, source_file = self.process_file(new_documents.pop())
            self.collection.add(
                documents=embeddings,
                metadatas=[{"source": source_file}],
                ids=[source_file]
            )
            metadata_store[source_file] = {
                "filename": source_file,
                "chunks": len(embeddings),
            }
            logger.info(f"Added document to collection: {source_file}")
        
        while docs_to_remove:
            doc_to_remove = docs_to_remove.pop()
            self.collection.delete(ids=[doc_to_remove])
            metadata_store.pop(doc_to_remove, None)
            logger.info(f"Removed document to collection: {doc_to_remove}")

        # Save updated metadata
        self.save_metadata(metadata_store)

        
        # vector_store = Chroma.from_documents(
        #     documents=chunks,
        #     embedding=embeddings,
        #     persist_directory=VECTOR_DB_PATH,
        # )
        self.collection.persist()
        # print(f"Vector store created at {VECTOR_DB_PATH}")
        return

In [124]:
manager = RAG_Manager()
# manager.get_files()
# manager.get_metadata()
manager.ingest_documents()

InternalError: Database error: error returned from database: (code: 1) no such table: tenants

In [30]:


logger = logging.getLogger(__name__)

# Constants (adjust these based on your setup)
METADATA_FILE = "../data/metadata.json"
EMBED_MODEL = "your_embedding_model"  # Replace with your actual model

class RAG_Manager:
    def __init__(self, data_path="../data", vector_db_path="../data/embeddings"):
        self.data_path = data_path
        self.vector_db_path = vector_db_path
        self.metadata_file = METADATA_FILE
        self.embedding_model = EMBED_MODEL
        self.embeddings = OllamaEmbeddings(model=self.embedding_model)
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=200,
            add_start_index=True,
        )
        os.makedirs(self.data_path, exist_ok=True)
        os.makedirs(os.path.dirname(self.metadata_file), exist_ok=True)
        
    def _clean_vector_db(self):
        """Clean the vector database directory to fix corruption issues"""
        if os.path.exists(self.vector_db_path):
            try:
                shutil.rmtree(self.vector_db_path)
                logger.info(f"Cleaned corrupted vector database at {self.vector_db_path}")
            except Exception as e:
                logger.error(f"Failed to clean vector database: {e}")
                
    def _load_metadata(self):
        if os.path.exists(self.metadata_file):
            try:
                with open(self.metadata_file, 'r') as f:
                    return json.load(f)
            except Exception as e:
                logger.warning(f"Metadata file not found or corrupted: {e}")
                return {}
        return {}
    
    def _save_metadata(self, metadata):
        with open(self.metadata_file, 'w') as f:
            json.dump(metadata, f, indent=4)
    
    def _get_documents(self):
        current_docs = {}
        if os.path.exists(self.data_path):
            for file in os.listdir(self.data_path):
                if file.endswith(".pdf"):
                    file_path = os.path.join(self.data_path, file)
                    current_docs[file] = {
                        'path': file_path                    
                    }
        return current_docs

    def _get_changes(self):
        stored_metadata = self._load_metadata()
        current_docs = self._get_documents()
        
        # Documents to add (new or modified)
        to_add = []
        # Documents to remove (deleted from filesystem)
        to_remove = []
        # Documents that are unchanged
        unchanged = []

        # Fix: Use proper iteration over dictionary items
        for doc_name, doc_info in current_docs.items():
            if doc_name not in stored_metadata:
                to_add.append(doc_name)
            else:
                unchanged.append(doc_name)
                
        for doc_name, doc_info in stored_metadata.items():
            if doc_name not in current_docs:
                to_remove.append(doc_name)
        
        return to_add, to_remove, unchanged
    
    def _get_chunks(self, doc_name, doc_path):
        try: 
            loader = PyPDFLoader(doc_path)
            documents = loader.load()
            chunks = self.text_splitter.split_documents(documents)
            for chunk in chunks:
                chunk.metadata['source_document'] = doc_name
            return chunks  # Fix: Return the chunks
        except Exception as e:
            logger.error(f"Error processing {doc_name}: {str(e)}")
            return []

    def _remove_documents_from_vector_store(self, vector_store, doc_names):
        for doc_name in doc_names:
            try:
                # Get all document IDs for this source document
                results = vector_store.get(
                    where={"source_document": doc_name}
                )
                if results['ids']:
                    vector_store.delete(ids=results['ids'])
                    logger.info(f"Removed {len(results['ids'])} chunks for document: {doc_name}")
                    
            except Exception as e:
                logger.error(f"Error removing document {doc_name}: {str(e)}")

    def _initialize_vector_store(self):
        """Initialize vector store with proper error handling"""
        max_retries = 3
        
        for attempt in range(max_retries):
            try:
                if os.path.exists(self.vector_db_path):
                    try:
                        # Try to load existing vector store
                        vector_store = Chroma(
                            persist_directory=self.vector_db_path,
                            embedding_function=self.embeddings
                        )
                        # Test the connection by trying to get collection info
                        _ = vector_store._collection.count()
                        logger.info("Successfully loaded existing vector store")
                        return vector_store
                    except Exception as e:
                        logger.warning(f"Failed to load existing vector store (attempt {attempt + 1}): {e}")
                        if attempt < max_retries - 1:
                            # Clean the corrupted database and try again
                            self._clean_vector_db()
                            continue
                        else:
                            raise e
                
                # Create new vector store
                logger.info("Creating new vector store")
                
                # Method 1: Try creating with a single dummy document
                try:
                    from langchain.schema import Document
                    dummy_doc = Document(
                        page_content="dummy content",
                        metadata={"source": "dummy"}
                    )
                    vector_store = Chroma.from_documents(
                        documents=[dummy_doc],
                        embedding=self.embeddings,
                        persist_directory=self.vector_db_path,
                    )
                    # Remove the dummy document
                    vector_store.delete(ids=vector_store.get()['ids'])
                    logger.info("Created new vector store with dummy document method")
                    return vector_store
                    
                except Exception as e:
                    logger.warning(f"Dummy document method failed: {e}")
                    
                    # Method 2: Try creating with ChromaDB client directly
                    try:
                        # Create ChromaDB client with proper settings
                        client_settings = Settings(
                            persist_directory=self.vector_db_path,
                            anonymized_telemetry=False
                        )
                        
                        # Create the vector store using the client
                        vector_store = Chroma(
                            persist_directory=self.vector_db_path,
                            embedding_function=self.embeddings,
                            client_settings=client_settings
                        )
                        logger.info("Created new vector store with direct client method")
                        return vector_store
                        
                    except Exception as e:
                        logger.error(f"Direct client method failed: {e}")
                        if attempt < max_retries - 1:
                            # Clean and try again
                            self._clean_vector_db()
                            continue
                        else:
                            raise e
                            
            except Exception as e:
                if attempt == max_retries - 1:
                    logger.error(f"All attempts to initialize vector store failed: {e}")
                    raise e
                else:
                    logger.warning(f"Attempt {attempt + 1} failed, retrying...")
                    self._clean_vector_db()

    def ingest_documents(self):
        try:
            vector_store = self._initialize_vector_store()
            to_add, to_remove, unchanged = self._get_changes()
            
            print(f"Documents to add: {to_add}")
            print(f"Documents to remove: {to_remove}")
            
            total_operations = len(to_add) + len(to_remove)
            current_operation = 0
            
            # Remove deleted documents
            if to_remove:
                self._remove_documents_from_vector_store(vector_store, to_remove)
                current_operation += len(to_remove)
                
            # Process new/modified documents
            all_new_chunks = []
            current_docs = self._get_documents()

            for i, doc_name in enumerate(to_add):
                if doc_name in current_docs:
                    # Process new version
                    doc_path = current_docs[doc_name]['path']
                    chunks = self._get_chunks(doc_name, doc_path)
                    
                    if chunks:
                        all_new_chunks.extend(chunks)
                        logger.info(f"Processed {doc_name}: {len(chunks)} chunks")
                    else:
                        logger.warning(f"No chunks created for {doc_name}")
            
            # Add new chunks to vector store
            if all_new_chunks:
                vector_store.add_documents(all_new_chunks)
                logger.info(f"Added {len(all_new_chunks)} chunks to vector store")
            
            # Persist vector store
            vector_store.persist()
            
            # Update metadata
            stored_metadata = self._load_metadata()
            
            # Remove metadata for deleted documents
            for doc_name in to_remove:
                if doc_name in stored_metadata:
                    del stored_metadata[doc_name]
            
            # Update metadata for new/modified documents
            for doc_name in to_add:
                if doc_name in current_docs:
                    doc_info = current_docs[doc_name].copy()
                    doc_info['processed_at'] = datetime.now().isoformat()
                    stored_metadata[doc_name] = doc_info
            
            self._save_metadata(stored_metadata)
            
            result = {
                'success': True,
                'total_documents': len(stored_metadata),
                'added': len(to_add),
                'removed': len(to_remove),
                'unchanged': len(unchanged),
                'total_chunks': len(all_new_chunks),
                'processed_at': datetime.now().isoformat()
            }
            
            logger.info(f"Ingestion completed: {result}")
            return result
            
        except Exception as e:
            logger.error(f"Document ingestion failed: {str(e)}")
            return {
                'success': False,
                'error': str(e),
                'total_documents': 0,
                'added': 0,
                'removed': 0,
                'unchanged': 0,
                'total_chunks': 0
            }

# Additional utility function for manual cleanup
def reset_vector_database(vector_db_path="../data/embeddings"):
    """Utility function to completely reset the vector database"""
    if os.path.exists(vector_db_path):
        try:
            shutil.rmtree(vector_db_path)
            print(f"Successfully reset vector database at {vector_db_path}")
        except Exception as e:
            print(f"Failed to reset vector database: {e}")
    else:
        print("Vector database directory doesn't exist")

# Usage example with error handling
if __name__ == "__main__":
    try:
        # Optional: Reset the database if you're having persistent issues
        # reset_vector_database()
        
        manager = RAG_Manager()
        result = manager.ingest_documents()
        print(f"\nIngestion result: {result}")
        
    except Exception as e:
        print(f"Failed to run ingestion: {e}")
        print("Try running reset_vector_database() first")

2025-07-08 12:14:40,128 - INFO - Creating new vector store
2025-07-08 12:14:40,130 - WARNING - Dummy document method failed: Database error: error returned from database: (code: 1) no such table: databases
2025-07-08 12:14:40,142 - INFO - Created new vector store with direct client method
2025-07-08 12:14:40,143 - INFO - Ingestion completed: {'success': True, 'total_documents': 0, 'added': 0, 'removed': 0, 'unchanged': 0, 'total_chunks': 0, 'processed_at': '2025-07-08T12:14:40.143778'}


Documents to add: []
Documents to remove: []

Ingestion result: {'success': True, 'total_documents': 0, 'added': 0, 'removed': 0, 'unchanged': 0, 'total_chunks': 0, 'processed_at': '2025-07-08T12:14:40.143778'}


In [31]:
# import shutil

# shutil.rmtree("../data/embeddings", ignore_errors=True)

manager = RAG_Manager()
result = manager.ingest_documents()
print(f"\nIngestion result: {result}")

2025-07-08 12:14:45,853 - INFO - Creating new vector store
2025-07-08 12:14:45,855 - WARNING - Dummy document method failed: Database error: error returned from database: (code: 1) no such table: databases
2025-07-08 12:14:45,857 - INFO - Created new vector store with direct client method
2025-07-08 12:14:45,858 - INFO - Ingestion completed: {'success': True, 'total_documents': 0, 'added': 0, 'removed': 0, 'unchanged': 0, 'total_chunks': 0, 'processed_at': '2025-07-08T12:14:45.858831'}


Documents to add: []
Documents to remove: []

Ingestion result: {'success': True, 'total_documents': 0, 'added': 0, 'removed': 0, 'unchanged': 0, 'total_chunks': 0, 'processed_at': '2025-07-08T12:14:45.858831'}


In [ ]:


# Ensure directories exist and are writable
os.makedirs(DATA_PATH, exist_ok=True)
os.makedirs(VECTOR_DB_PATH, exist_ok=True)

# Verify write permissions
if not os.access(VECTOR_DB_PATH, os.W_OK):
    raise PermissionError(f"No write permission for {VECTOR_DB_PATH}")

def ingest_documents():
    documents = []
    for file in sorted(os.listdir(DATA_PATH)):
        if file.endswith(".pdf"):
            print(os.path.join(DATA_PATH, file))
            loader = PyPDFLoader(os.path.join(DATA_PATH, file))
            documents.extend(loader.load())
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        add_start_index=True,
    )
    chunks = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(chunks)} chunks.")

    embeddings = OllamaEmbeddings(model="nomic-embed-text") # Or mxbai-embed-large
    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=VECTOR_DB_PATH,
    )
    vector_store.persist()
    print(f"Vector store created at {VECTOR_DB_PATH}")

In [3]:
ingest_documents()

../data/documents/ Benjamin Graham, David Dodd, Seth A. Klarman - Security Analysis, Seventh Edition_ Principles and Techniques (2023, McGraw Hill) - libgen.li.pdf
../data/documents/ Bernstein, Peter L - Against the Gods_ the Remarkable Story of Risk (2012, Wiley) - libgen.li.pdf
../data/documents/ Brown, Stephen_Elton, Edwin_Goetzmann, William_Gruber, Martin - Modern Portfolio Theory and Investment Analysis (2014_2013, Wiley) - libgen.li.pdf
../data/documents/ Burton G. Malkiel - A Random Walk Down Wall Street - libgen.li.pdf
../data/documents/ GRAHAM, Benjamin - The Intelligent Investor_ A Book of Practical Counsel (2003, HarperCollins Publishers, Inc. _ PerfectBound) - libgen.li.pdf
../data/documents/[Wiley investment classics] Fisher, Philip A - Common stocks and uncommon profits and other writings by Philip A. Fisher (1996, Wiley) - libgen.li.pdf
Split 3890 documents into 11477 chunks.


/var/folders/gs/2xrc2py11l77xsmgngltxlvr0000gn/T/ipykernel_40579/1381229644.py:30: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="nomic-embed-text") # Or mxbai-embed-large


Vector store created at ../data/embeddings


/var/folders/gs/2xrc2py11l77xsmgngltxlvr0000gn/T/ipykernel_40579/1381229644.py:36: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vector_store.persist()
